In [ ]:
# SECTION 1: Imports and configuration
from __future__ import annotations

import csv, hashlib, json, os, re, sys, time
from dataclasses import dataclass, field
from itertools import combinations
from pathlib import Path
from types import SimpleNamespace
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage.feature import hog
from skimage.morphology import skeletonize

try:
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision as mp_vision
except ImportError:
    mp = mp_python = mp_vision = None
    print("[warn] THERE IS NO MEDIAPIPE ")

try:
    import tensorflow as tf
except ImportError:
    tf = None

# Configuration

def _base_dir() -> Path:
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()

BASE_DIR = _base_dir()

@dataclass
class Config:
    hand_model: str = os.environ.get(
        "HAND_MODEL", str(BASE_DIR / "hand_landmarker.task")
    )
    siamese_weights: str = os.environ.get(
        "SIGNATURE_WEIGHTS", str(BASE_DIR / "signature_encoder.weights.h5")
    )
    enroll_dir: Path = BASE_DIR / "biometric_2fa_data" / "air_enrollments"

    camera_index: int = 0
    camera_width: int = 1280
    camera_height: int = 720
    camera_fps: int = 30
    detect_width: int = 960

    enroll_count: int = 5
    trajectory_points: int = 128

    img_w: int = 155
    img_h: int = 220

    stroke_color: Tuple[int, int, int] = (0, 0, 0)
    stroke_halo: Tuple[int, int, int] = (255, 255, 255)
    stroke_thickness: int = 5

    palm_stable_frames: int = 5
    palm_cooldown_frames: int = 25
    tail_trim_points: int = 6
    banner_frames: int = 60

    consistency_max_ratio: float = 2.2
    stroke_count_tolerance: int = 3
    stroke_count_penalty: float = 0.95

    w_dtw: float = 0.50
    w_embed: float = 0.20
    w_hog: float = 0.18
    w_hu: float = 0.12

    threshold_floor: float = 0.35
    threshold_ceiling: float = 0.85
    threshold_margin_std: float = 2.6

    # ---- 2FA login ----
    login_max_tries: int = 3
    ident_margin: float = 0.02
    show_scores: bool = False

# ---------------- Fingerprint pipeline config (same values as Harman_BIOM_2) ----------------
FP_CFG = SimpleNamespace(
    BLOCK        = 16,
    ROI_MARGIN_P = 1.5,
    MIN_COH      = 0.25,
    MIN_QUALITY  = 0.30,
    SPUR_PERIODS = 1.0,
    N_ORIENT     = 16,
    TRACE_STEPS  = 10,
    DEFAULT_RIDGE_FREQ = 1/9,
)

# ---------------- 2FA system config ----------------
SYS = SimpleNamespace(
    DATA_DIR = BASE_DIR / "biometric_2fa_data",

    # --- fingerprint enrolment / matching ---
    FP_MIN_ENROLL_SAMPLES  = 2,
    FP_MIN_MINUTIAE        = 12,
    FP_MIN_IMAGE_QUALITY   = 40.0,
    FP_THRESHOLD           = 0.10,
    FP_MIN_PAIRS           = 6,
    FP_ENROLL_CONSISTENCY  = 0.6,
    FP_MAX_ROTATION_DEG    = 60,
    FP_TOL_PERIODS         = 1.2,
    FP_TOL_ANGLE_DEG       = 25,

    # --- login security ---
    MAX_FAILS   = 3,
    LOCKOUT_SEC = 60,
    DEBUG       = True,
)

SIG_CFG = Config()
SYS.DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Data folder :", SYS.DATA_DIR)

In [ ]:
# SECTION 2: Air signature - hand tracking, gesture recognition, stroke capture
# Geometry helpers

def landmark_xy(hand, idx: int) -> np.ndarray:
    p = hand[idx]
    return np.array([p.x, p.y], dtype=np.float32)

def landmark_xyz(hand, idx: int) -> np.ndarray:
    p = hand[idx]
    return np.array([p.x, p.y, p.z], dtype=np.float32)

def joint_angle(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    ba, bc = a - b, c - b
    denom = float(np.linalg.norm(ba) * np.linalg.norm(bc))
    if denom < 1e-8:
        return 0.0
    cos = np.clip(np.dot(ba, bc) / denom, -1.0, 1.0)
    return float(np.degrees(np.arccos(cos)))

def polyline_length(points: np.ndarray) -> np.ndarray:
    if len(points) < 2:
        return np.zeros(len(points), dtype=np.float32)
    seg = np.linalg.norm(np.diff(points, axis=0), axis=1)
    return np.concatenate(([0.0], np.cumsum(seg))).astype(np.float32)

def resample(points: np.ndarray, n: int) -> np.ndarray:
    points = np.asarray(points, dtype=np.float32)
    if len(points) == 0:
        return points
    if len(points) < 2:
        return np.repeat(points[:1], n, axis=0)
    s = polyline_length(points)
    if s[-1] < 1e-8:
        return np.repeat(points[:1], n, axis=0)
    t = np.linspace(0.0, float(s[-1]), n)
    return np.column_stack(
        (np.interp(t, s, points[:, 0]), np.interp(t, s, points[:, 1]))
    ).astype(np.float32)

def gaussian_smooth(points: np.ndarray, radius: int = 2, sigma: float = 1.0) -> np.ndarray:
    if len(points) < 2 * radius + 1:
        return points.copy()
    x = np.arange(-radius, radius + 1, dtype=np.float32)
    k = np.exp(-(x * x) / (2 * sigma * sigma))
    k /= k.sum()
    padded = np.pad(points, ((radius, radius), (0, 0)), mode="edge")
    out = np.empty_like(points)
    for d in range(points.shape[1]):
        out[:, d] = np.convolve(padded[:, d], k, mode="valid")
    return out

# Gesture recognition

FINGERS = {
    "index": (8, 7, 6, 5),
    "middle": (12, 11, 10, 9),
    "ring": (16, 15, 14, 13),
    "pinky": (20, 19, 18, 17),
}

def finger_extended(hand, tip: int, dip: int, pip: int, mcp: int) -> bool:
    a_top = joint_angle(landmark_xyz(hand, pip), landmark_xyz(hand, dip), landmark_xyz(hand, tip))
    a_mid = joint_angle(landmark_xyz(hand, mcp), landmark_xyz(hand, pip), landmark_xyz(hand, dip))
    wrist = landmark_xy(hand, 0)
    tip_d = np.linalg.norm(landmark_xy(hand, tip) - wrist)
    pip_d = np.linalg.norm(landmark_xy(hand, pip) - wrist)
    return a_top >= 140.0 and a_mid >= 140.0 and tip_d > pip_d * 1.05

PINCH_ON = 0.45    
PINCH_OFF = 0.72   

def pinch_ratio(hand) -> float:
    gap = np.linalg.norm(landmark_xy(hand, 4) - landmark_xy(hand, 8))
    palm = np.linalg.norm(landmark_xy(hand, 0) - landmark_xy(hand, 9)) + 1e-6
    return float(gap / palm)

def pen_point(hand) -> np.ndarray:
    """Pen tip = midpoint between thumb and index fingertips."""
    return (landmark_xy(hand, 4) + landmark_xy(hand, 8)) / 2

def finger_curled(hand, tip: int, pip: int) -> bool:
    wrist = landmark_xy(hand, 0)
    return np.linalg.norm(landmark_xy(hand, tip) - wrist) < np.linalg.norm(landmark_xy(hand, pip) - wrist)

def detect_gesture(hand, was_pinching: bool) -> str:
    ratio = pinch_ratio(hand)

    if was_pinching:
        if ratio < PINCH_OFF:
            return "PINCH"

    index_curled = finger_curled(hand, 8, 6)
    is_fist_shape = index_curled and finger_curled(hand, 12, 10)

    if not is_fist_shape and ratio < PINCH_ON:
        return "PINCH"
    ext = {name: finger_extended(hand, *ids) for name, ids in FINGERS.items()}
    count = sum(ext.values())
    if count == 4:
        return "PALM"
    if count == 2 and ext["index"] and ext["middle"]:
        return "UNDO"
    if count == 0 and is_fist_shape:
        return "FIST"

    return "NONE"

class GestureStabilizer:
    """Debounces raw per-frame gestures into a stable state."""

    def __init__(self, palm_frames: int):
        self.required = {"PINCH": 2, "PALM": palm_frames, "FIST": 6, "UNDO": 8, "NONE": 2}
        self.current = "NONE"
        self.candidate = "NONE"
        self.count = 0
        self.missing = 0

    def update(self, detected: str) -> str:
        if detected == "LOST":
            self.missing += 1
            if self.missing <= 4:     
                return self.current
            detected = "NONE"
        else:
            self.missing = 0

        if detected == self.candidate:
            self.count += 1
        else:
            self.candidate = detected
            self.count = 1

        if self.count >= self.required.get(detected, 3):
            self.current = detected
        return self.current

# Fingertip smoothing

class OneEuroFilter:
    """Low-lag adaptive smoothing (Casiez et al.)."""

    def __init__(self, min_cutoff: float = 1.6, beta: float = 0.6, d_cutoff: float = 1.0):
        self.min_cutoff, self.beta, self.d_cutoff = min_cutoff, beta, d_cutoff
        self.reset()

    def reset(self) -> None:
        self.x_prev: Optional[np.ndarray] = None
        self.dx_prev: Optional[np.ndarray] = None
        self.t_prev: Optional[float] = None

    @staticmethod
    def _alpha(cutoff: float, dt: float) -> float:
        tau = 1.0 / (2.0 * np.pi * max(cutoff, 1e-6))
        return 1.0 / (1.0 + tau / max(dt, 1e-6))

    def __call__(self, x: np.ndarray, t: float) -> np.ndarray:
        x = np.asarray(x, dtype=np.float64)
        if self.x_prev is None:
            self.x_prev, self.dx_prev, self.t_prev = x.copy(), np.zeros_like(x), t
            return x.astype(np.float32)
        dt = float(np.clip(t - self.t_prev, 1e-3, 0.1))
        dx = (x - self.x_prev) / dt
        a_d = self._alpha(self.d_cutoff, dt)
        dx_hat = a_d * dx + (1 - a_d) * self.dx_prev
        cutoff = self.min_cutoff + self.beta * float(np.linalg.norm(dx_hat))
        a = self._alpha(cutoff, dt)
        x_hat = a * x + (1 - a) * self.x_prev
        self.x_prev, self.dx_prev, self.t_prev = x_hat, dx_hat, t
        return x_hat.astype(np.float32)

class TipTracker:

    MAX_REJECTS = 2
    BASE_STEP = 0.20
    CAP_STEP = 0.45

    def __init__(self):
        self.filter = OneEuroFilter()
        self.prev_raw: Optional[np.ndarray] = None
        self.prev_out: Optional[np.ndarray] = None
        self.prev_t = 0.0
        self.velocity = np.zeros(2, dtype=np.float32)
        self.rejects = 0
        self.pending_bridge: List[np.ndarray] = []

    def reset(self) -> None:
        self.filter.reset()
        self.prev_raw = None
        self.prev_out = None
        self.velocity[:] = 0
        self.rejects = 0
        self.pending_bridge = []

    def _seed(self, xy: np.ndarray, t: float, bridge_from: Optional[np.ndarray] = None) -> np.ndarray:
        self.filter.reset()
        out = self.filter(xy, t)

        if bridge_from is not None:
            n = 4
            self.pending_bridge = [bridge_from + (out - bridge_from) * (k / n) for k in range(1, n)]
        self.prev_raw, self.prev_out, self.prev_t = xy.copy(), out.copy(), t
        self.velocity[:] = 0
        self.rejects = 0
        return out

    def update(self, xy: np.ndarray, t: float) -> Tuple[np.ndarray, bool]:
        xy = np.asarray(xy, dtype=np.float32)
        if self.prev_raw is None:
            return self._seed(xy, t), True

        dt = float(np.clip(t - self.prev_t, 1e-3, 0.1))
        step = float(np.linalg.norm(xy - self.prev_raw))
        predicted = float(np.linalg.norm(self.velocity) * dt)
        max_step = float(np.clip(self.BASE_STEP + 2.5 * predicted, self.BASE_STEP, self.CAP_STEP))

        if step > max_step:
            self.rejects += 1
            if self.rejects <= self.MAX_REJECTS:
                return self.prev_out.copy(), False
            last_out = self.prev_out.copy()
            return self._seed(xy, t, bridge_from=last_out), True

        self.rejects = 0
        out = self.filter(xy, t)
        self.velocity = 0.6 * self.velocity + 0.4 * (xy - self.prev_raw) / dt
        self.prev_raw, self.prev_out, self.prev_t = xy.copy(), out.copy(), t
        return out, True

# Stroke capture

class StrokeCapture:
    MIN_POINT_DIST = 0.0015

    def __init__(self):
        self.strokes: List[np.ndarray] = []
        self.current: List[np.ndarray] = []
        self.tracker = TipTracker()

    def start_stroke(self) -> None:
        self.current = []
        self.tracker.reset()

    def add(self, xy: np.ndarray, t: float) -> Optional[np.ndarray]:
        point, accepted = self.tracker.update(xy, t)
        if not accepted:
            return point

        for b in self.tracker.pending_bridge:
            if not self.current or np.linalg.norm(b - self.current[-1]) >= self.MIN_POINT_DIST:
                self.current.append(b.copy())
        self.tracker.pending_bridge = []
        if self.current and np.linalg.norm(point - self.current[-1]) < self.MIN_POINT_DIST:
            return point
        self.current.append(point.copy())
        return point

    def finish_stroke(self, trim: int = 0) -> None:
        pts = self.current[: len(self.current) - trim] if trim else self.current
        if len(pts) == 1:
            pts = [pts[0], pts[0] + np.float32(0.002)]
        if len(pts) >= 2:
            self.strokes.append(np.asarray(pts, dtype=np.float32))
        self.current = []
        self.tracker.reset()

    def clear(self) -> None:
        self.strokes = []
        self.current = []
        self.tracker.reset()

    def undo(self) -> bool:
        if self.current:
            self.current = []
            self.tracker.reset()
            return True
        if self.strokes:
            self.strokes.pop()
            return True
        return False

    def all_strokes(self) -> List[np.ndarray]:
        out = list(self.strokes)
        if len(self.current) >= 2:
            out.append(np.asarray(self.current, dtype=np.float32))
        return out

    def point_count(self) -> int:
        return sum(len(s) for s in self.all_strokes())

In [ ]:
# SECTION 3: Air signature - trajectory normalization, features, matcher, storage, camera
# Trajectory normalisation and rendering

def normalize_strokes(strokes: Sequence[np.ndarray], n_points: int) -> Optional[np.ndarray]:
    valid = [np.asarray(s, dtype=np.float32) for s in strokes if len(s) >= 2]
    if not valid:
        return None
    pts = np.concatenate(valid, axis=0)
    lo, hi = pts.min(axis=0), pts.max(axis=0)
    span = float(max(hi - lo))
    if span < 1e-6:
        return None

    lengths = np.array([max(float(polyline_length(s)[-1]), 1e-6) for s in valid])
    alloc = np.maximum(6, np.round(n_points * lengths / lengths.sum())).astype(int)
    pieces = [gaussian_smooth(resample((s - lo) / span, int(n))) for s, n in zip(valid, alloc)]
    traj = resample(np.concatenate(pieces, axis=0), n_points)

    lo2, hi2 = traj.min(axis=0), traj.max(axis=0)
    span2 = float(max(max(hi2 - lo2), 1e-6))
    centre = (lo2 + hi2) / 2
    traj = (traj - centre) / span2 + 0.5
    return np.clip(traj, 0, 1).astype(np.float32)

def _render_polylines(polys: Sequence[np.ndarray], width: int, height: int,
                      thickness: int, margin: int) -> np.ndarray:
    """Black-on-white raster of unit-box polylines, aspect ratio preserved,
    supersampled for smooth anti-aliased strokes."""
    canvas = np.full((height, width), 255, dtype=np.uint8)
    polys = [p for p in polys if len(p) >= 2]
    if not polys:
        return canvas
    box = min(width, height) - 2 * margin
    offset = ((width - box) / 2, (height - box) / 2)
    scale = 4
    big = np.full((height * scale, width * scale), 255, dtype=np.uint8)
    pts = [((p * box + offset) * scale).round().astype(np.int32).reshape(-1, 1, 2) for p in polys]
    cv2.polylines(big, pts, False, 0, thickness * scale, cv2.LINE_AA)
    return cv2.resize(big, (width, height), interpolation=cv2.INTER_AREA)

def render_signature(traj: np.ndarray, width: int, height: int,
                     thickness: int = 4, margin: int = 12) -> np.ndarray:
    """Raster of a normalised trajectory (used for CNN / HOG / Hu)."""
    if traj is None:
        traj = np.empty((0, 2), dtype=np.float32)
    return _render_polylines([traj], width, height, thickness, margin)

def render_strokes(strokes: Sequence[np.ndarray], width: int, height: int,
                   thickness: int = 6, margin: int = 30) -> np.ndarray:
    """Human-readable raster of raw strokes (pen-up gaps left blank)."""
    valid = [np.asarray(s, dtype=np.float32) for s in strokes if len(s) >= 2]
    if not valid:
        return _render_polylines([], width, height, thickness, margin)
    pts = np.concatenate(valid, axis=0)
    lo, hi = pts.min(axis=0), pts.max(axis=0)
    span = float(max(max(hi - lo), 1e-6))
    centre = (lo + hi) / 2
    polys = [gaussian_smooth((s - centre) / span + 0.5) for s in valid]
    return _render_polylines(polys, width, height, thickness, margin)

def preprocess_image(img: np.ndarray, width: int, height: int) -> np.ndarray:
    """Binary float image (ink = 1) for the CNN / HOG / Hu features."""
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    binary = cv2.resize(binary, (width, height), interpolation=cv2.INTER_AREA)
    return (binary.astype(np.float32) / 255.0)[..., None]

# Features

def dtw_distance(a: np.ndarray, b: np.ndarray, band: int = 12) -> float:
    """Sakoe-Chiba banded DTW between two (n, d) sequences."""
    n, m = len(a), len(b)
    cost = np.linalg.norm(a[:, None, :] - b[None, :, :], axis=2)
    acc = np.full((n + 1, m + 1), np.inf, dtype=np.float64)
    acc[0, 0] = 0.0
    for i in range(1, n + 1):
        j0, j1 = max(1, i - band), min(m, i + band)
        for j in range(j0, j1 + 1):
            acc[i, j] = cost[i - 1, j - 1] + min(acc[i - 1, j], acc[i, j - 1], acc[i - 1, j - 1])
    return float(acc[n, m] / (n + m))

def trajectory_descriptor(traj: np.ndarray) -> np.ndarray:
    """Position + unit direction per point (n, 4)."""
    d = np.gradient(traj, axis=0)
    d /= np.linalg.norm(d, axis=1, keepdims=True) + 1e-8
    return np.concatenate([traj, 0.5 * d], axis=1).astype(np.float32)

def hog_feature(img: np.ndarray) -> np.ndarray:
    f = hog(img[..., 0], orientations=9, pixels_per_cell=(11, 11),
            cells_per_block=(2, 2), block_norm="L2-Hys").astype(np.float32)
    return f / (np.linalg.norm(f) + 1e-8)

def hu_feature(img: np.ndarray) -> np.ndarray:
    m = cv2.moments((img[..., 0] * 255).astype(np.uint8))
    hu = cv2.HuMoments(m).flatten()
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    return np.nan_to_num(hu).astype(np.float32)

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

class SiameseEncoder:
    """Optional CNN embedding. Silently disabled if TF/weights are missing."""

    def __init__(self, weights: str, img_w: int, img_h: int):
        self.model = None
        if tf is None:
            print("[info] tensorflow not installed - Siamese embedding disabled")
            return
        if not os.path.isfile(weights):
            print(f"[info] Siamese weights not found ({weights}) - embedding disabled")
            return
        try:
            self.model = self._build(img_w, img_h)
            self.model.load_weights(weights)
            print("[info] Siamese encoder loaded")
        except Exception as exc:
            self.model = None
            print(f"[warn] could not load Siamese weights ({exc}) - embedding disabled")

    @staticmethod
    def _build(img_w: int, img_h: int):
        L = tf.keras.layers
        inputs = tf.keras.Input(shape=(img_h, img_w, 1))
        x = inputs
        for filters, k, pool in ((32, 5, True), (64, 3, True), (96, 3, True), (128, 3, False)):
            x = L.Conv2D(filters, k, padding="same")(x)
            x = L.BatchNormalization()(x)
            x = L.ReLU()(x)
            if pool:
                x = L.MaxPooling2D(2)(x)
        x = L.GlobalAveragePooling2D()(x)
        x = L.Dense(256, activation="relu")(x)
        x = L.Dropout(0.25)(x)
        x = L.Dense(128)(x)
        outputs = L.Lambda(lambda z: tf.math.l2_normalize(z, axis=1))(x)
        return tf.keras.Model(inputs, outputs, name="signature_encoder")

    @property
    def available(self) -> bool:
        return self.model is not None

    def embed(self, img: np.ndarray) -> Optional[np.ndarray]:
        if self.model is None:
            return None
        e = self.model.predict(img[None], verbose=0)[0]
        return (e / (np.linalg.norm(e) + 1e-8)).astype(np.float32)

# Matching

@dataclass
class Sample:
    traj: np.ndarray
    desc: np.ndarray
    hog: np.ndarray
    hu: np.ndarray
    embed: Optional[np.ndarray]

@dataclass
class Profile:
    samples: List[Sample]
    dtw_scale: float
    threshold: float
    weights: Dict[str, float] = field(default_factory=dict)
    stroke_counts: List[int] = field(default_factory=list)
    consistency: float = 0.0

class SignatureMatcher:
    def __init__(self, cfg: Config, encoder: SiameseEncoder):
        self.cfg = cfg
        self.encoder = encoder

    # ---- feature extraction ----
    def make_sample(self, traj: np.ndarray) -> Sample:
        img = preprocess_image(render_signature(traj, self.cfg.img_w, self.cfg.img_h),
                               self.cfg.img_w, self.cfg.img_h)
        return Sample(
            traj=traj,
            desc=trajectory_descriptor(traj),
            hog=hog_feature(img),
            hu=hu_feature(img),
            embed=self.encoder.embed(img),
        )

    def _weights(self) -> Dict[str, float]:
        w = {"dtw": self.cfg.w_dtw, "embed": self.cfg.w_embed,
             "hog": self.cfg.w_hog, "hu": self.cfg.w_hu}
        if not self.encoder.available:
            w["embed"] = 0.0
        total = sum(w.values())
        return {k: v / total for k, v in w.items()}

    # ---- pairwise score ----
    def pair_score(self, q: Sample, r: Sample, dtw_scale: float, w: Dict[str, float]) -> float:
        s_dtw = float(np.exp(-dtw_distance(q.desc, r.desc) / max(dtw_scale, 1e-6)))
        s_hog = (cosine(q.hog, r.hog) + 1) / 2
        s_hu = float(np.exp(-np.linalg.norm(q.hu - r.hu) / 3.0))
        score = w["dtw"] * s_dtw + w["hog"] * s_hog + w["hu"] * s_hu
        if w["embed"] > 0 and q.embed is not None and r.embed is not None:
            score += w["embed"] * (cosine(q.embed, r.embed) + 1) / 2
        return score

    # ---- calibration from enrolment ----
    def consistency_ratio(self, traj: np.ndarray, prev: Sequence[np.ndarray]) -> float:
        """How far a new sample is from the previous ones, relative to how
        far they are from each other (1.0 = as consistent as the rest)."""
        if len(prev) < 2:
            return 1.0
        descs = [trajectory_descriptor(p) for p in prev]
        q = trajectory_descriptor(traj)
        n = len(descs)
        mutual = np.median([dtw_distance(descs[i], descs[j])
                            for i in range(n) for j in range(i + 1, n)])
        to_new = np.median([dtw_distance(q, d) for d in descs])
        return float(to_new / max(mutual, 1e-6))

    def build_profile(self, trajs: Sequence[np.ndarray],
                      stroke_counts: Sequence[int] = ()) -> Profile:
        samples = [self.make_sample(t) for t in trajs]
        w = self._weights()
        n = len(samples)
        dists = [dtw_distance(samples[i].desc, samples[j].desc)
                 for i in range(n) for j in range(i + 1, n)]

        dtw_scale = 2.0 * float(np.median(dists)) if dists else 0.1
        dtw_scale = max(dtw_scale, 0.02)

        loo = []
        for i in range(n):
            others = [s for j, s in enumerate(samples) if j != i]
            if others:
                loo.append(self._top_k_mean(samples[i], others, dtw_scale, w))
        if loo:
            thr = float(np.mean(loo) - self.cfg.threshold_margin_std * (np.std(loo) + 0.02))

            if len(loo) <= 2:
                thr -= 0.10
        else:
            thr = self.cfg.threshold_floor

        thr = float(np.clip(thr, self.cfg.threshold_floor, self.cfg.threshold_ceiling))
        return Profile(samples=samples, dtw_scale=dtw_scale, threshold=thr, weights=w,
                       stroke_counts=list(stroke_counts),
                       consistency=float(np.mean(loo)) if loo else 0.0)

    def _top_k_mean(self, q: Sample, refs: Sequence[Sample], dtw_scale: float,
                    w: Dict[str, float], k: int = 3) -> float:
        scores = sorted((self.pair_score(q, r, dtw_scale, w) for r in refs), reverse=True)
        return float(np.mean(scores[:k]))

    # ---- verification ----
    def verify_sample(self, q: Sample, profile: Profile,
                      stroke_count: Optional[int] = None) -> Tuple[bool, float]:
        """Match an already-extracted sample against a profile (IDENTIFY extracts features only once)."""
        score = self._top_k_mean(q, profile.samples, profile.dtw_scale, profile.weights)
        if stroke_count is not None and profile.stroke_counts:
            expected = int(np.median(profile.stroke_counts))
            if abs(stroke_count - expected) > self.cfg.stroke_count_tolerance:
                score *= self.cfg.stroke_count_penalty
        return score >= profile.threshold, score

    def verify(self, traj: np.ndarray, profile: Profile,
               stroke_count: Optional[int] = None) -> Tuple[bool, float]:
        return self.verify_sample(self.make_sample(traj), profile, stroke_count)

# Storage

class EnrollmentStore:
    def __init__(self, root: Path, user_id: str):
        self.dir = root / user_id
        self.meta = self.dir / "profile.json"

    def count(self) -> int:
        return len(list(self.dir.glob("signature_*.npy"))) if self.dir.exists() else 0

    def reset(self) -> int:
        """Remove previous samples so a fresh enrolment starts clean."""
        removed = 0
        if self.dir.exists():
            for p in list(self.dir.glob("signature_*.*")) + [self.meta]:
                if p.is_file():
                    p.unlink()
                    removed += 1
        return removed

    def save_sample(self, traj: np.ndarray, image: np.ndarray, index: int,
                    stroke_count: int = 0) -> Path:
        self.dir.mkdir(parents=True, exist_ok=True)
        np.save(self.dir / f"signature_{index:02d}.npy", traj)
        png = self.dir / f"signature_{index:02d}.png"
        cv2.imwrite(str(png), image)
        (self.dir / f"signature_{index:02d}.json").write_text(json.dumps({
            "strokes": stroke_count,
            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }))
        return png

    def load_trajectories(self) -> List[np.ndarray]:
        return [t for t, _ in self.load_samples()]

    def load_samples(self) -> List[Tuple[np.ndarray, Optional[int]]]:
        """(trajectory, stroke_count) pairs; stroke_count is None for
        samples saved by older versions."""
        out = []
        for p in sorted(self.dir.glob("signature_*.npy")) if self.dir.exists() else []:
            t = np.load(p)
            if t.ndim != 2 or t.shape[1] != 2 or len(t) < 2:
                continue
            meta = p.with_suffix(".json")
            strokes = None
            if meta.is_file():
                try:
                    strokes = int(json.loads(meta.read_text()).get("strokes", 0)) or None
                except (ValueError, json.JSONDecodeError):
                    strokes = None
            out.append((t.astype(np.float32), strokes))
        return out

    def save_calibration(self, profile: Profile) -> None:
        self.meta.write_text(json.dumps({
            "version": 2,
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "threshold": profile.threshold,
            "dtw_scale": profile.dtw_scale,
            "weights": profile.weights,
            "samples": len(profile.samples),
            "stroke_counts": profile.stroke_counts,
            "consistency": profile.consistency,
        }, indent=2))

    def log_verification(self, ok: bool, score: float, threshold: float, strokes: int) -> None:
        path = self.dir / "verify_log.csv"
        new = not path.is_file()
        with path.open("a", encoding="utf-8") as f:
            if new:
                f.write("time,result,score,threshold,strokes\n")
            f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')},{'MATCH' if ok else 'NO_MATCH'},"
                    f"{score:.4f},{threshold:.4f},{strokes}\n")

    def load_calibration(self) -> Optional[dict]:
        if self.meta.is_file():
            return json.loads(self.meta.read_text())
        return None

# Camera

def open_camera(cfg: Config) -> cv2.VideoCapture:
    backends = [cv2.CAP_DSHOW, cv2.CAP_MSMF, cv2.CAP_ANY] if os.name == "nt" else [cv2.CAP_ANY]
    for backend in backends:
        cap = cv2.VideoCapture(cfg.camera_index, backend)
        if cap.isOpened():
            cap.set(cv2.CAP_PROP_FRAME_WIDTH, cfg.camera_width)
            cap.set(cv2.CAP_PROP_FRAME_HEIGHT, cfg.camera_height)
            cap.set(cv2.CAP_PROP_FPS, cfg.camera_fps)
            cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
            ok, _ = cap.read()
            if ok:
                return cap
        cap.release()
    raise RuntimeError(f"Camera {cfg.camera_index} could not be opened")

In [ ]:
# SECTION 4: AirSignatureApp - ENROLL / VERIFY / IDENTIFY (returns a result dict)

class AirSignatureApp:
    WINDOW = "Air Signature"

    def __init__(self, cfg: Config, user_id: Optional[str], mode: str,
                 profiles: Optional[Dict[str, Profile]] = None,
                 matcher: Optional[SignatureMatcher] = None):
        """mode: ENROLL | VERIFY (claimed user_id) | IDENTIFY (user_id=None, profiles={uid: Profile})"""
        self.cfg = cfg
        self.user_id = user_id
        self.mode = mode
        self.profiles = profiles or {}
        if matcher is None:
            matcher = SignatureMatcher(cfg, SiameseEncoder(cfg.siamese_weights, cfg.img_w, cfg.img_h))
        self.matcher = matcher
        self.encoder = matcher.encoder
        self.store = EnrollmentStore(cfg.enroll_dir, user_id) if user_id else None
        self.profile: Optional[Profile] = None
        self.tries_left = cfg.login_max_tries
        self.finished = False
        self.result: dict = {"ok": False}

        self.capture = StrokeCapture()
        self.stabilizer = GestureStabilizer(cfg.palm_stable_frames)
        self.prev_gesture = "NONE"
        self.palm_cooldown = 0
        self.saved = 0
        self.status = "Pinch thumb + index to draw"
        self.status_color = (255, 255, 255)
        self.fps = 0.0
        self.last_ts = 0
        self.pinch = 1.0
        self.banner: Optional[Tuple[str, Tuple[int, int, int]]] = None
        self.banner_left = 0
        self.thumb: Optional[np.ndarray] = None

        if mode == "VERIFY":
            self._load_profile()
        elif mode == "IDENTIFY":
            if not self.profiles:
                raise RuntimeError("No enrolled signature profiles found for IDENTIFY")
        elif self.store.reset():
            print(f"[info] previous enrolment for '{user_id}' removed - starting fresh")

    # ---------------- setup ----------------
    def _build_profile(self) -> Profile:
        samples = self.store.load_samples()
        if not samples:
            raise RuntimeError(f"No enrolled signatures for '{self.user_id}'. Enroll first.")
        trajs = [t for t, _ in samples]
        counts = [c for _, c in samples if c is not None]
        return self.matcher.build_profile(trajs, counts)

    def _load_profile(self) -> None:
        self.profile = self._build_profile()
        print(f"[info] loaded {len(self.profile.samples)} signatures, "
              f"threshold={self.profile.threshold:.3f}, consistency={self.profile.consistency:.2f}")

    def _make_landmarker(self):
        if mp is None:
            raise RuntimeError("mediapipe is not installed: pip install mediapipe")
        options = mp_vision.HandLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=self.cfg.hand_model),
            running_mode=mp_vision.RunningMode.VIDEO,
            num_hands=1,

            min_hand_detection_confidence=0.45,
            min_hand_presence_confidence=0.45,
            min_tracking_confidence=0.45,
        )
        return mp_vision.HandLandmarker.create_from_options(options)

    # ---------------- actions ----------------
    def _reset_canvas(self, message: str, color=(255, 255, 255)) -> None:
        self.capture.clear()
        self.status, self.status_color = message, color

    def _show_banner(self, text: str, color: Tuple[int, int, int]) -> None:
        self.banner, self.banner_left = (text, color), self.cfg.banner_frames

    def _on_palm(self) -> None:
        if self.finished:
            return
        if self.capture.current:
            self.capture.finish_stroke(trim=self.cfg.tail_trim_points)
        strokes = self.capture.all_strokes()
        if len(strokes) == 0 or self.capture.point_count() < 12:
            self.status, self.status_color = "Draw a signature first", (0, 200, 255)
            return
        traj = normalize_strokes(strokes, self.cfg.trajectory_points)
        if traj is None:
            self._reset_canvas("Signature too small - draw again", (0, 200, 255))
            return
        self.palm_cooldown = self.cfg.palm_cooldown_frames
        if self.mode == "ENROLL":
            self._enroll(traj, strokes)
        else:
            self._verify(traj, len(strokes))

    def _enroll(self, traj: np.ndarray, strokes: Sequence[np.ndarray]) -> None:
        prev = self.store.load_trajectories()
        ratio = self.matcher.consistency_ratio(traj, prev)
        if ratio > self.cfg.consistency_max_ratio:
            msg = f"Too different from previous samples (x{ratio:.1f}) - draw it again"
            print(f"[enroll] rejected: {msg}")
            self._reset_canvas(msg, (0, 200, 255))
            self._show_banner("DRAW AGAIN", (0, 200, 255))
            return

        self.saved += 1
        image = render_strokes(strokes, 400, 400)
        self.store.save_sample(traj, image, self.saved, stroke_count=len(strokes))
        self.thumb = cv2.cvtColor(cv2.resize(image, (140, 140), interpolation=cv2.INTER_AREA),
                                  cv2.COLOR_GRAY2BGR)
        msg = f"SAVED {self.saved}/{self.cfg.enroll_count}  ({len(strokes)} strokes)"
        print(f"[enroll] {msg}")
        self._show_banner(f"SAVED {self.saved}/{self.cfg.enroll_count}", (80, 255, 80))
        if self.saved >= self.cfg.enroll_count:
            profile = self._build_profile()
            self.store.save_calibration(profile)
            msg = (f"ENROLLMENT COMPLETE  threshold {profile.threshold:.2f}  "
                   f"consistency {profile.consistency:.2f}")
            print(f"[enroll] {msg}")
            self._show_banner("ENROLLED", (80, 255, 80))
            self.result = dict(ok=True, samples=len(profile.samples),
                               threshold=float(profile.threshold),
                               consistency=float(profile.consistency))
            self.finished = True
        self._reset_canvas(msg, (80, 255, 80))

    def _identify(self, traj: np.ndarray, stroke_count: int):
        """1:N match against all enrolled users; accept only if exactly one user clearly passes."""
        q = self.matcher.make_sample(traj)
        cands = []
        for uid, prof in self.profiles.items():
            ok, score = self.matcher.verify_sample(q, prof, stroke_count)
            cands.append((score, uid, ok, prof.threshold))
        cands.sort(reverse=True)
        passing = [c for c in cands if c[2]]
        if not passing:
            return False, None, cands[0][0], cands[0][3], "no profile matched"
        if len(passing) > 1 and passing[0][0] - passing[1][0] < self.cfg.ident_margin:
            return False, None, passing[0][0], passing[0][3], "ambiguous (two users too close)"
        score, uid, _, thr = passing[0]
        return True, uid, score, thr, "ok"

    def _verify(self, traj: np.ndarray, stroke_count: int) -> None:
        if self.mode == "IDENTIFY":
            ok, who, score, thr, why = self._identify(traj, stroke_count)
        else:
            assert self.profile is not None
            ok, score = self.matcher.verify(traj, self.profile, stroke_count)
            who, thr, why = self.user_id, self.profile.threshold, ("ok" if ok else "below threshold")
            self.store.log_verification(ok, score, thr, stroke_count)
        detail = f"  score {score:.3f} / thr {thr:.2f}" if self.cfg.show_scores else ""
        print(f"[{self.mode.lower()}] {'MATCH' if ok else 'NO MATCH'}{detail}")
        color = (80, 255, 80) if ok else (80, 80, 255)
        self._show_banner("MATCH" if ok else "NO MATCH", color)
        if ok:
            self.result = dict(ok=True, user_id=who, score=float(score), threshold=float(thr))
            self.finished = True
            self._reset_canvas("Signature accepted" + detail, color)
            return
        self.tries_left -= 1
        self.result = dict(ok=False, user_id=None, score=float(score), threshold=float(thr), reason=why)
        if self.tries_left <= 0:
            self.finished = True
            self._reset_canvas("Signature rejected" + detail, color)
        else:
            self._reset_canvas(f"NO MATCH - {self.tries_left} tries left" + detail, color)

    # ---------------- drawing ----------------
    def _draw_strokes(self, frame: np.ndarray) -> None:
        h, w = frame.shape[:2]
        polys = []
        for s in self.capture.all_strokes():
            pts = (s * (w, h)).round().astype(np.int32).reshape(-1, 1, 2)
            polys.append(pts)
        if not polys:
            return
        t = self.cfg.stroke_thickness
        cv2.polylines(frame, polys, False, self.cfg.stroke_halo, t + 4, cv2.LINE_AA)
        cv2.polylines(frame, polys, False, self.cfg.stroke_color, t, cv2.LINE_AA)

    def _draw_hud(self, frame: np.ndarray, gesture: str, hand) -> None:
        h, w = frame.shape[:2]
        if hand is not None:
            tip = pen_point(hand)
            colour = (0, 0, 255) if gesture == "PINCH" else (0, 220, 0)
            cv2.circle(frame, (int(tip[0] * w), int(tip[1] * h)), 9, colour, 2, cv2.LINE_AA)

        bar = frame[0:70]
        cv2.addWeighted(bar, 0.35, np.full_like(bar, (25, 25, 25)), 0.65, 0, bar)
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(frame, f"AIR SIGNATURE  |  {self.mode}  |  user: {self.user_id or '?'}",
                    (16, 28), font, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        info = f"gesture {gesture:<5}  fps {self.fps:4.1f}"
        if self.mode == "ENROLL":
            info += f"   saved {self.saved}/{self.cfg.enroll_count}"
        elif self.profile is not None:
            info += f"   references {len(self.profile.samples)}"
        cv2.putText(frame, info, (16, 56), font, 0.55, (200, 230, 255), 1, cv2.LINE_AA)

        mx, my, mw, mh = w - 176, 22, 150, 14
        fill = int(mw * float(np.clip(1.0 - self.pinch / 1.2, 0.0, 1.0)))
        cv2.rectangle(frame, (mx, my), (mx + mw, my + mh), (90, 90, 90), -1)
        cv2.rectangle(frame, (mx, my), (mx + fill, my + mh),
                      (0, 0, 255) if gesture == "PINCH" else (0, 200, 120), -1)
        on_x = mx + int(mw * (1.0 - PINCH_ON / 1.2))
        cv2.line(frame, (on_x, my - 3), (on_x, my + mh + 3), (255, 255, 255), 1)
        cv2.putText(frame, "pinch", (mx, my + mh + 16), font, 0.45, (200, 230, 255), 1, cv2.LINE_AA)

        if self.thumb is not None:
            th, tw = self.thumb.shape[:2]
            y0, x0 = 84, w - tw - 16
            frame[y0:y0 + th, x0:x0 + tw] = self.thumb
            cv2.rectangle(frame, (x0 - 1, y0 - 1), (x0 + tw, y0 + th), (200, 200, 200), 1)
            cv2.putText(frame, "last saved", (x0, y0 + th + 18), font, 0.45, (200, 230, 255), 1, cv2.LINE_AA)

        if self.banner is not None and self.banner_left > 0:
            text, color = self.banner
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_DUPLEX, 2.2, 5)
            cx, cy = (w - tw) // 2, (h + th) // 2
            overlay = frame.copy()
            cv2.rectangle(overlay, (cx - 40, cy - th - 30), (cx + tw + 40, cy + 30), (25, 25, 25), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
            cv2.putText(frame, text, (cx, cy), cv2.FONT_HERSHEY_DUPLEX, 2.2, color, 5, cv2.LINE_AA)
            self.banner_left -= 1

        foot = frame[h - 64:h]
        cv2.addWeighted(foot, 0.35, np.full_like(foot, (25, 25, 25)), 0.65, 0, foot)
        cv2.putText(frame, self.status[:80], (16, h - 38), font, 0.6, self.status_color, 2, cv2.LINE_AA)
        cv2.putText(frame, "PINCH = draw (release = pause)   PALM = save / verify   V = undo   FIST = clear   Q / ESC = quit",
                    (16, h - 12), font, 0.5, (220, 220, 220), 1, cv2.LINE_AA)

    # ---------------- main loop ----------------
    def run(self) -> None:
        landmarker = self._make_landmarker()
        cap = open_camera(self.cfg)
        cv2.namedWindow(self.WINDOW, cv2.WINDOW_AUTOSIZE)
        prev_time = time.perf_counter()
        failed = 0
        try:
            while True:
                ok, frame = cap.read()
                if not ok:
                    failed += 1
                    if failed > 30:
                        raise RuntimeError("Camera stopped delivering frames")
                    continue
                failed = 0
                frame = cv2.flip(frame, 1)

                now = time.perf_counter()
                inst = 1.0 / max(now - prev_time, 1e-6)
                prev_time = now
                self.fps = inst if self.fps == 0 else 0.9 * self.fps + 0.1 * inst

                hand = self._detect(landmarker, frame)
                self.pinch = pinch_ratio(hand) if hand is not None else 1.0
                raw = detect_gesture(hand, self.prev_gesture == "PINCH") if hand is not None else "LOST"
                gesture = self.stabilizer.update(raw)
                self._handle_gesture(gesture, hand, time.monotonic())

                self._draw_strokes(frame)
                self._draw_hud(frame, gesture, hand)
                cv2.imshow(self.WINDOW, frame)

                if cv2.waitKey(1) & 0xFF in (ord("q"), 27):
                    break
                self.prev_gesture = gesture
                if self.finished and self.banner_left <= 0:
                    break
        finally:
            cap.release()
            cv2.destroyAllWindows()
            landmarker.close()
            print(f"\n{'=' * 60}\n{self.mode} finished"
                  + (f": {self.saved}/{self.cfg.enroll_count} saved" if self.mode == "ENROLL" else "")
                  + f"\n{'=' * 60}")
        return self.result

    def _detect(self, landmarker, frame: np.ndarray):
        h, w = frame.shape[:2]
        if w > self.cfg.detect_width:
            scale = self.cfg.detect_width / w
            small = cv2.resize(frame, (self.cfg.detect_width, int(h * scale)), interpolation=cv2.INTER_AREA)
        else:
            small = frame
        rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
        image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
        ts = max(int(time.monotonic_ns() // 1_000_000), self.last_ts + 1)
        self.last_ts = ts
        result = landmarker.detect_for_video(image, ts)
        return result.hand_landmarks[0] if result.hand_landmarks else None

    def _handle_gesture(self, gesture: str, hand, t: float) -> None:
        started = gesture != self.prev_gesture

        if gesture == "PINCH":
            if started:
                self.capture.start_stroke()
            if hand is not None:
                self.capture.add(pen_point(hand), t)
        elif self.prev_gesture == "PINCH":
            self.capture.finish_stroke(trim=self.cfg.tail_trim_points if gesture == "PALM" else 0)

        if gesture == "PALM" and started and self.palm_cooldown == 0:
            self._on_palm()
        elif gesture == "FIST" and started:
            self._reset_canvas("Cleared")
        elif gesture == "UNDO" and started:
            if self.capture.undo():
                self.status, self.status_color = "Last stroke removed", (255, 255, 255)

        if self.palm_cooldown > 0:
            self.palm_cooldown -= 1

In [ ]:
# SECTION 5: Fingerprint minutiae extraction pipeline
def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return img

def segment(gray, block):
    g = gray.astype(np.float32)
    mean = cv2.blur(g, (block, block))
    sq   = cv2.blur(g * g, (block, block))
    std  = np.sqrt(np.maximum(sq - mean * mean, 0))

    std8 = cv2.normalize(std, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    t, _ = cv2.threshold(std8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (std8 > 0.6 * t).astype(np.uint8)

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (block, block))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)
    m = ndi.binary_fill_holes(m).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)

    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n > 1:
        big = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        m = (lab == big).astype(np.uint8)
    m = cv2.erode(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (block + 1, block + 1)))
    return m

def fix_polarity(gray, mask, min_bg_frac=0.05):
    bg = gray[mask == 0]
    if bg.size > min_bg_frac * gray.size:
        if bg.mean() < gray[mask > 0].mean():
            return 255 - gray, True
    return gray, False

def normalize(gray, mask, sigma=12):
    g = gray.astype(np.float32)
    w = mask.astype(np.float32)
    den = cv2.GaussianBlur(w, (0, 0), sigma) + 1e-6
    mu  = cv2.GaussianBlur(g * w, (0, 0), sigma) / den
    var = cv2.GaussianBlur(((g - mu) ** 2) * w, (0, 0), sigma) / den
    gstd = g[mask > 0].std() + 1e-6
    out = (g - mu) / (np.sqrt(var) + 0.2 * gstd)
    out[mask == 0] = 0
    return out

def orientation_field(norm, mask, block):
    sm = cv2.GaussianBlur(norm, (0, 0), 1.0)
    gx = cv2.Sobel(sm, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(sm, cv2.CV_32F, 0, 1, ksize=3)

    s = block * 0.6
    Gxx = cv2.GaussianBlur(gx * gx, (0, 0), s)
    Gyy = cv2.GaussianBlur(gy * gy, (0, 0), s)
    Gxy = cv2.GaussianBlur(gx * gy, (0, 0), s)

    num = 2 * Gxy
    den = Gxx - Gyy
    mag = np.sqrt(num ** 2 + den ** 2)
    coherence = (mag / (Gxx + Gyy + 1e-6)).astype(np.float32)
    coherence[mask == 0] = 0
    
    c = cv2.GaussianBlur(den / (mag + 1e-6), (0, 0), block * 0.4)
    s2 = cv2.GaussianBlur(num / (mag + 1e-6), (0, 0), block * 0.4)
    theta_grad = 0.5 * np.arctan2(s2, c)
    ridge_theta = np.mod(theta_grad + np.pi / 2, np.pi).astype(np.float32)
    return ridge_theta, coherence

def estimate_ridge_frequency(norm, mask, default_f, B=32, step=16, nfft=128):
    h, w = norm.shape
    win = np.outer(np.hanning(B), np.hanning(B)).astype(np.float32)
    fx = np.fft.fftshift(np.fft.fftfreq(nfft))
    r = np.hypot(*np.meshgrid(fx, fx))
    band = (r >= 1 / 25) & (r <= 1 / 4)
    freqs = []
    for y in range(0, h - B, step):
        for x in range(0, w - B, step):
            if mask[y:y + B, x:x + B].mean() < 0.95:
                continue
            patch = norm[y:y + B, x:x + B] * win
            F = np.abs(np.fft.fftshift(np.fft.fft2(patch, s=(nfft, nfft))))
            Fb = np.where(band, F, 0)
            idx = np.argmax(Fb)
            if Fb.flat[idx] > 3 * (F[band].mean() + 1e-6):
                freqs.append(r.flat[idx])
    if len(freqs) < 5:
        return default_f
    return float(np.median(freqs))

def make_gabor(phi, freq, s_across, s_along):
    half = int(np.ceil(2.5 * max(s_across, s_along)))
    y, x = np.mgrid[-half:half + 1, -half:half + 1].astype(np.float32)
    c, s = np.cos(phi), np.sin(phi)
    v = x * c + y * s
    u = -x * s + y * c
    g = np.exp(-0.5 * (u ** 2 / s_across ** 2 + v ** 2 / s_along ** 2)) * np.cos(2 * np.pi * freq * u)
    g -= g.mean()
    g /= np.abs(g).sum()
    return g.astype(np.float32)

def gabor_enhance(norm, ridge_theta, mask, freq, n_orient):
    period = 1.0 / freq
    s_across, s_along = 0.45 * period, 0.8 * period
    resp = np.empty((n_orient,) + norm.shape, np.float32)
    for k in range(n_orient):
        phi = k * np.pi / n_orient
        resp[k] = cv2.filter2D(norm, cv2.CV_32F, make_gabor(phi, freq, s_across, s_along),
                               borderType=cv2.BORDER_REFLECT)
    pos = ridge_theta / (np.pi / n_orient)
    k0 = np.floor(pos).astype(int) % n_orient
    k1 = (k0 + 1) % n_orient
    a = (pos - np.floor(pos)).astype(np.float32)
    r0 = np.take_along_axis(resp, k0[None], 0)[0]
    r1 = np.take_along_axis(resp, k1[None], 0)[0]
    out = (1 - a) * r0 + a * r1
    out[mask == 0] = 0
    return out

def remove_small_blobs(b, min_area):
    n, lab, stats, _ = cv2.connectedComponentsWithStats(b.astype(np.uint8), connectivity=8)
    keep = stats[:, cv2.CC_STAT_AREA] >= min_area
    keep[0] = False
    return keep[lab].astype(np.uint8)

def binarize_ridges(enh, mask, period):
    ridge = ((enh < 0) & (mask > 0)).astype(np.uint8)    
    amin = max(8, int(0.7 * period * period))
    ridge = remove_small_blobs(ridge, amin)                       
    ridge = 1 - remove_small_blobs(1 - ridge, amin)           
    return ridge * (mask > 0)

_RING = [(-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1)]

def crossing_number(sk):
    p = np.pad(sk.astype(np.int8), 1)
    H, W = sk.shape
    ring = [p[1 + dy:1 + dy + H, 1 + dx:1 + dx + W] for dy, dx in _RING]
    cn = np.zeros(sk.shape, np.int8)
    for i in range(8):
        cn += np.abs(ring[i] - ring[(i + 1) % 8])
    return cn // 2

def prune_spurs(sk, max_len, passes=2):
    sk = sk.astype(np.uint8).copy()
    H, W = sk.shape
    for _ in range(passes):
        cn = crossing_number(sk)
        ends = np.argwhere((sk == 1) & (cn == 1))
        changed = False
        for y, x in ends:
            if sk[y, x] == 0:
                continue
            path, cur, hit_junction = [(y, x)], (y, x), False
            while len(path) <= max_len:
                cy, cx = cur
                nb = [(cy + dy, cx + dx) for dy, dx in _RING
                      if 0 <= cy + dy < H and 0 <= cx + dx < W
                      and sk[cy + dy, cx + dx] and (cy + dy, cx + dx) not in path]
                if not nb:
                    break                                  
                if len(nb) > 1 or cn[nb[0]] >= 3:
                    hit_junction = True
                    break
                cur = nb[0]
                path.append(cur)
            if len(path) <= max_len:          
                for py, px in path:
                    sk[py, px] = 0
                changed = True
        if not changed:
            break
    return sk

def _branch_starts(sk, y, x):
    H, W = sk.shape
    vals = [1 if (0 <= y + dy < H and 0 <= x + dx < W and sk[y + dy, x + dx]) else 0 for dy, dx in _RING]
    starts = []
    for i in range(8):
        if vals[i] and not vals[i - 1]:           
            run = [i]
            j = (i + 1) % 8
            while vals[j] and j != i:
                run.append(j)
                j = (j + 1) % 8
            mid = run[len(run) // 2]
            starts.append((y + _RING[mid][0], x + _RING[mid][1]))
    return starts

def _trace(sk, origin, start, steps):
    H, W = sk.shape
    oy, ox = origin
    visited = {origin} | {(oy + dy, ox + dx) for dy, dx in _RING}
    cur, n = start, 1
    for _ in range(steps - 1):
        cy, cx = cur
        nxt = [(cy + dy, cx + dx) for dy, dx in _RING
               if 0 <= cy + dy < H and 0 <= cx + dx < W and sk[cy + dy, cx + dx]
               and (cy + dy, cx + dx) not in visited]
        if not nxt:
            break
        cur = nxt[0]
        visited.add(cur)
        n += 1
    return cur, n

def minutia_direction(sk, x, y, kind, steps):
    starts = _branch_starts(sk, y, x)
    angs, lens = [], []
    for st in starts:
        end, n = _trace(sk, (y, x), st, steps)
        angs.append(np.arctan2(end[0] - y, end[1] - x))
        lens.append(n)
    if not angs:
        return 0.0, 0
    if kind == "Ending" or len(angs) < 3:
        return float(angs[0]), int(lens[0])
    best, pair = 10, (0, 1)
    for i in range(len(angs)):
        for j in range(i + 1, len(angs)):
            d = abs((angs[i] - angs[j] + np.pi) % (2 * np.pi) - np.pi)
            if d < best:
                best, pair = d, (i, j)
    a, b = angs[pair[0]], angs[pair[1]]
    return float(np.arctan2(np.sin(a) + np.sin(b), np.cos(a) + np.cos(b))), int(min(lens))

def _clip01(v):
    return float(np.clip(v, 0.0, 1.0))

def extract_minutiae(sk, mask, coherence, clarity, period, cfg):
    H, W = sk.shape
    cn = crossing_number(sk)
    dt = cv2.distanceTransform(np.pad(mask, 1), cv2.DIST_L2, 5)[1:-1, 1:-1]
    margin = cfg.ROI_MARGIN_P * period

    pts = np.argwhere((sk == 1) & ((cn == 1) | (cn == 3)) & (dt > margin))
    out = []
    for y, x in pts:
        kind = "Ending" if cn[y, x] == 1 else "Bifurcation"
        ang, tl = minutia_direction(sk, x, y, kind, cfg.TRACE_STEPS)
        q_coh   = _clip01(coherence[y, x] / 0.8)
        q_clar  = _clip01(clarity[y, x])
        q_dist  = _clip01(dt[y, x] / (3 * period))
        q_trace = _clip01(tl / cfg.TRACE_STEPS)
        q = 0.40 * q_coh + 0.30 * q_clar + 0.15 * q_dist + 0.15 * q_trace
        out.append(dict(x=int(x), y=int(y), type=kind, angle=ang, quality=q,
                        coherence=float(coherence[y, x]), border_dist=float(dt[y, x])))
    return out

def remove_false_minutiae(mins, period, cfg):
    stats = {"raw": len(mins)}
    mins = [m for m in mins if m["coherence"] >= cfg.MIN_COH and m["quality"] >= cfg.MIN_QUALITY]
    stats["after_quality"] = len(mins)
    if len(mins) < 2:
        stats["final"] = len(mins)
        return mins, stats

    P = period
    xy = np.array([[m["x"], m["y"]] for m in mins], float)
    tree = cKDTree(xy)
    drop = set()

    for i, j in sorted(tree.query_pairs(1.5 * P)):
        a, b = mins[i], mins[j]
        d = np.hypot(a["x"] - b["x"], a["y"] - b["y"])
        ta, tb = a["type"], b["type"]
        v = np.array([b["x"] - a["x"], b["y"] - a["y"]], float)

        if d < 0.4 * P:                                         
            drop.add(i if a["quality"] < b["quality"] else j)
        elif ta == tb == "Ending" and d < 1.5 * P:
            da = np.array([np.cos(a["angle"]), np.sin(a["angle"])])
            db = np.array([np.cos(b["angle"]), np.sin(b["angle"])])
            if da @ v < 0 and db @ (-v) < 0:                     
                drop.update((i, j))                                
        elif ta != tb and d < 0.7 * P:                             
            drop.update((i, j))
        elif ta == tb == "Bifurcation" and d < 0.8 * P:     
            drop.update((i, j))

    kept = [m for k, m in enumerate(mins) if k not in drop]
    stats["after_pair_rules"] = len(kept)
    if len(kept) > 3:
        xy = np.array([[m["x"], m["y"]] for m in kept], float)
        cnt = np.array([len(n) - 1 for n in cKDTree(xy).query_ball_point(xy, 2.0 * P)])
        kept = [m for m, c in zip(kept, cnt) if c <= 4]
    stats["final"] = len(kept)
    return kept, stats

def _wrap(d):
    return (d + np.pi / 2) % np.pi - np.pi / 2
def detect_singularities(ridge_theta, coherence, mask, block, min_coh=0.2):
    H, W = ridge_theta.shape
    rows, cols = H // block, W // block
    if rows < 6 or cols < 6:
        return [], []
    sl = (slice(0, rows * block), slice(0, cols * block))
    c = cv2.resize(np.cos(2 * ridge_theta)[sl], (cols, rows), interpolation=cv2.INTER_AREA)
    s = cv2.resize(np.sin(2 * ridge_theta)[sl], (cols, rows), interpolation=cv2.INTER_AREA)
    c, s = cv2.GaussianBlur(c, (3, 3), 0.8), cv2.GaussianBlur(s, (3, 3), 0.8)
    th = 0.5 * np.arctan2(s, c)
    coh = cv2.resize(coherence[sl], (cols, rows), interpolation=cv2.INTER_AREA)
    roi = cv2.resize(mask[sl].astype(np.float32), (cols, rows), interpolation=cv2.INTER_AREA) > 0.98
    roi = cv2.erode(roi.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)

    cores, deltas = [], []
    for i in range(2, rows - 2):
        for j in range(2, cols - 2):
            if not roi[i - 2:i + 3, j - 2:j + 3].all():
                continue
            if coh[i - 2:i + 3, j - 2:j + 3].mean() < min_coh:
                continue
            ring = [th[i - 1, j - 1], th[i - 1, j], th[i - 1, j + 1], th[i, j + 1],
                    th[i + 1, j + 1], th[i + 1, j], th[i + 1, j - 1], th[i, j - 1], th[i - 1, j - 1]]
            tot = sum(_wrap(ring[k + 1] - ring[k]) for k in range(8))
            if abs(tot - np.pi) < 0.35:
                cores.append((j, i))
            elif abs(tot + np.pi) < 0.35:
                deltas.append((j, i))

    def nms(pts, d=3):
        kept = []
        for p in pts:
            if all(np.hypot(p[0] - q[0], p[1] - q[1]) >= d for q in kept):
                kept.append(p)
        return kept

    to_px = lambda pts: [(x * block + block // 2, y * block + block // 2) for x, y in nms(pts)]
    return to_px(cores), to_px(deltas)

def process_fingerprint(gray, cfg=FP_CFG):
    r = {}
    mask = segment(gray, cfg.BLOCK)
    gray, inverted = fix_polarity(gray, mask)
    norm = normalize(gray, mask)

    ridge_theta, coherence = orientation_field(norm, mask, cfg.BLOCK)
    freq = estimate_ridge_frequency(norm, mask, cfg.DEFAULT_RIDGE_FREQ)
    period = 1.0 / freq

    enh = gabor_enhance(norm, ridge_theta, mask, freq, cfg.N_ORIENT)
    binary = binarize_ridges(enh, mask, period)
    skel = skeletonize(binary).astype(np.uint8)
    skel[mask == 0] = 0
    skel = prune_spurs(skel, max_len=max(4, int(cfg.SPUR_PERIODS * period)))

    amp = cv2.GaussianBlur(np.abs(enh), (0, 0), cfg.BLOCK * 0.5)
    ref = np.percentile(amp[mask > 0], 90) + 1e-6 if mask.any() else 1.0
    clarity = np.clip(amp / ref, 0, 1)

    raw = extract_minutiae(skel, mask, coherence, clarity, period, cfg)
    final, fstats = remove_false_minutiae(raw, period, cfg)
    cores, deltas = detect_singularities(ridge_theta, coherence, mask, cfg.BLOCK)

    roi = mask > 0
    roi_area = int(roi.sum())
    mean_coh = float(coherence[roi].mean()) if roi_area else 0.0
    mean_clar = float(clarity[roi].mean()) if roi_area else 0.0
    good_area = float((coherence[roi] > 0.3).mean()) if roi_area else 0.0
    n = len(final)
    n_end = sum(m["type"] == "Ending" for m in final)
    n_bif = n - n_end
    q_img = 100 * (0.35 * _clip01(mean_coh / 0.6) + 0.25 * mean_clar
                   + 0.25 * good_area + 0.15 * _clip01(n / 20))
    grade = "Good" if q_img >= 65 else ("Fair" if q_img >= 40 else "Poor")

    scores = dict(
        n_minutiae=n, n_endings=n_end, n_bifurcations=n_bif,
        end_bif_ratio=round(n_end / max(1, n_bif), 2),
        mean_minutia_quality=round(float(np.mean([m["quality"] for m in final])) if final else 0.0, 3),
        mean_coherence=round(mean_coh, 3), mean_clarity=round(mean_clar, 3),
        good_area_ratio=round(good_area, 3),
        roi_area_ratio=round(roi_area / mask.size, 3),
        ridge_period_px=round(period, 2),
        n_core=len(cores), n_delta=len(deltas),
        image_quality=round(q_img, 1), quality_grade=grade,
        polarity_inverted=inverted,
    )
    r.update(gray=gray, mask=mask, norm=norm, ridge_theta=ridge_theta, coherence=coherence,
             enhanced=enh, binary=binary, skeleton=skel, raw_minutiae=raw, minutiae=final,
             filter_stats=fstats, cores=cores, deltas=deltas, period=period, scores=scores)
    return r

def show_result(res, title=""):
    g = res["gray"]
    s = res["scores"]

    R = 5
    T = 1      
    L = 18      

    fin = cv2.cvtColor(g, cv2.COLOR_GRAY2RGB)
    for m in res["minutiae"]:
        c = (0, 200, 0) if m["type"] == "Ending" else (255, 0, 0)
        cv2.circle(fin, (m["x"], m["y"]), R, c, T)
        cv2.line(fin, (m["x"], m["y"]),
                 (int(m["x"] + L * np.cos(m["angle"])),
                  int(m["y"] + L * np.sin(m["angle"]))), c, T)

    fig, ax = plt.subplots(1, 2, figsize=(20, 12))

    ax[0].imshow(g, cmap="gray")
    ax[0].set_title("Original", fontsize=16)

    ax[1].imshow(fin)
    ax[1].set_title(f"Minutiae: {s['n_endings']} endings (green) + "
                    f"{s['n_bifurcations']} bifurcations (red)", fontsize=16)

    for a in ax:
        a.axis("off")

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

In [ ]:
# SECTION 6: Fingerprint template + matcher (1:1 verification via minutiae alignment)
def _sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def build_fp_template(image_path, cfg=FP_CFG):
    """Image -> compact template (only minutiae + metadata; the raw image is not stored)."""
    res = process_fingerprint(load_gray(str(image_path)), cfg)
    s = res["scores"]
    return dict(
        source=os.path.basename(str(image_path)),
        path=str(Path(image_path).resolve()),
        sha256=_sha256_file(image_path),
        period=float(res["period"]),
        image_quality=float(s["image_quality"]),
        grade=s["quality_grade"],
        n_minutiae=int(s["n_minutiae"]),
        minutiae=[dict(x=int(m["x"]), y=int(m["y"]), type=m["type"],
                       angle=float(m["angle"]), quality=float(m["quality"])) for m in res["minutiae"]],
    )

def fp_quality_problems(tpl):
    problems = []
    if tpl["n_minutiae"] < SYS.FP_MIN_MINUTIAE:
        problems.append(f"only {tpl['n_minutiae']} minutiae found (at least {SYS.FP_MIN_MINUTIAE} required)")
    if tpl["image_quality"] < SYS.FP_MIN_IMAGE_QUALITY:
        problems.append(f"image quality {tpl['image_quality']:.0f} < {SYS.FP_MIN_IMAGE_QUALITY:.0f}")
    return problems

def _wrap_pi(a):
    return (a + np.pi) % (2 * np.pi) - np.pi

def _to_array(mins):
    """list[dict] -> array [x, y, angle, is_ending, quality]"""
    return np.array([[m["x"], m["y"], m["angle"], 1.0 if m["type"] == "Ending" else 0.0,
                      m.get("quality", 1.0)] for m in mins], dtype=np.float64).reshape(-1, 5)

def _candidate_transforms(Q, R, period, max_rot, n_cand=8):
    """Hough voting: candidate (rotation, tx, ty) transforms that align Q onto R."""
    tb, ab = period, np.radians(10)
    rows = []
    for t in (0.0, 1.0):
        q, r = Q[Q[:, 3] == t], R[R[:, 3] == t]
        if len(q) == 0 or len(r) == 0:
            continue
        dth = _wrap_pi(r[:, None, 2] - q[None, :, 2])
        c, s = np.cos(dth), np.sin(dth)
        tx = r[:, None, 0] - (q[None, :, 0] * c - q[None, :, 1] * s)
        ty = r[:, None, 1] - (q[None, :, 0] * s + q[None, :, 1] * c)
        ok = np.abs(dth) <= max_rot
        rows.append(np.column_stack([dth[ok], tx[ok], ty[ok]]))
    if not rows:
        return []
    V = np.vstack(rows)
    if len(V) == 0:
        return []
    K = np.column_stack([np.round(V[:, 1] / tb), np.round(V[:, 2] / tb), np.round(V[:, 0] / ab)]).astype(int)
    uk, cnt = np.unique(K, axis=0, return_counts=True)
    cands = []
    for idx in np.argsort(-cnt)[: n_cand * 3]:
        near = (np.abs(K - uk[idx]) <= 1).all(axis=1)
        cands.append((int(near.sum()), V[near].mean(axis=0)))
    cands.sort(key=lambda z: -z[0])
    out = []
    for _, m in cands:
        if all(abs(m[0] - o[0]) > ab / 2 or np.hypot(m[1] - o[1], m[2] - o[2]) > tb / 2 for o in out):
            out.append(m)
        if len(out) >= n_cand:
            break
    return out

def _apply(Q, th, tx, ty):
    c, s = np.cos(th), np.sin(th)
    out = Q.copy()
    out[:, 0] = Q[:, 0] * c - Q[:, 1] * s + tx
    out[:, 1] = Q[:, 0] * s + Q[:, 1] * c + ty
    out[:, 2] = _wrap_pi(Q[:, 2] + th)
    return out

def _pair(Qt, R, tol_d, tol_a):
    """One-to-one pairing (Hungarian) - same type, within distance and angle tolerance."""
    D = np.hypot(Qt[:, None, 0] - R[None, :, 0], Qt[:, None, 1] - R[None, :, 1])
    A = np.abs(_wrap_pi(Qt[:, None, 2] - R[None, :, 2]))
    T = Qt[:, None, 3] == R[None, :, 3]
    cost = np.where((D <= tol_d) & (A <= tol_a) & T, D, 1e6)
    ri, ci = linear_sum_assignment(cost)
    good = cost[ri, ci] < 1e5
    return ri[good], ci[good]

def _refine(Qt, R, ri, ci):
    """Re-estimate the rigid transform (rotation + shift) from matched pairs to align Q more tightly."""
    P, T = Qt[ri, :2], R[ci, :2]
    pc, tc = P.mean(0), T.mean(0)
    p, t = P - pc, T - tc
    dth = np.arctan2((p[:, 0] * t[:, 1] - p[:, 1] * t[:, 0]).sum(), (p * t).sum())
    c, s = np.cos(dth), np.sin(dth)
    out = Qt.copy()
    x, y = Qt[:, 0] - pc[0], Qt[:, 1] - pc[1]
    out[:, 0] = x * c - y * s + tc[0]
    out[:, 1] = x * s + y * c + tc[1]
    out[:, 2] = _wrap_pi(Qt[:, 2] + dth)
    return out

def _in_region(P, ref_xy, margin):
    """Count how many points of P lie inside the convex hull (+ margin) of the reference points (overlap area)."""
    if len(ref_xy) < 3:
        return len(P)
    hull = cv2.convexHull(ref_xy.astype(np.float32))
    return int(sum(cv2.pointPolygonTest(hull, (float(x), float(y)), True) >= -margin for x, y in P[:, :2]))

def match_minutiae(q_mins, r_mins, period,
                   max_rot_deg=None, tol_periods=None, tol_angle_deg=None):
    max_rot = np.radians(SYS.FP_MAX_ROTATION_DEG if max_rot_deg is None else max_rot_deg)
    tol_d = (SYS.FP_TOL_PERIODS if tol_periods is None else tol_periods) * period
    tol_a = np.radians(SYS.FP_TOL_ANGLE_DEG if tol_angle_deg is None else tol_angle_deg)

    best = dict(score=0.0, pairs=0, n_query=len(q_mins), n_ref=len(r_mins), rotation_deg=0.0)
    if len(q_mins) < 3 or len(r_mins) < 3:
        return best
    Q, R = _to_array(q_mins), _to_array(r_mins)

    for th, tx, ty in _candidate_transforms(Q, R, period, max_rot):
        Qt = _apply(Q, th, tx, ty)
        ri, ci = _pair(Qt, R, tol_d, tol_a)
        if len(ri) < 3:
            continue
        Qt = _refine(Qt, R, ri, ci)
        ri, ci = _pair(Qt, R, tol_d, tol_a)
        m = len(ri)
        if m < 3:
            continue
        nq = max(_in_region(Qt, R[:, :2], tol_d), m)
        nr = max(_in_region(R, Qt[:, :2], tol_d), m)
        score = (m * m) / float(nq * nr)
        if score > best["score"]:
            best.update(score=float(score), pairs=int(m), rotation_deg=float(np.degrees(th)),
                        n_query=int(nq), n_ref=int(nr))
    return best

def match_fp(q_tpl, r_tpl):
    period = float(np.median([q_tpl["period"], r_tpl["period"]]))
    return match_minutiae(q_tpl["minutiae"], r_tpl["minutiae"], period)

def fp_accept(result):
    return result["score"] >= SYS.FP_THRESHOLD and result["pairs"] >= SYS.FP_MIN_PAIRS

In [ ]:
# SECTION 7: User registry, lockout, audit log
def sanitize_user(user: str) -> str:
    cleaned = "".join(c if c.isalnum() or c in "_-" else "_" for c in (user or "").strip())
    if not cleaned:
        raise ValueError("user_id cannot be empty")
    return cleaned

class Registry:
    """users.json = {users: {id: {...}}, lockouts: {key: {...}}}
    status: PENDING (one factor still missing) | ACTIVE (both factors enrolled -> login allowed)."""

    def __init__(self, root: Path):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.path = self.root / "users.json"
        self.audit_path = self.root / "audit_log.csv"

    # ---- storage ----
    def _load(self) -> dict:
        if self.path.is_file():
            return json.loads(self.path.read_text(encoding="utf-8"))
        return {"users": {}, "lockouts": {}}

    def _save(self, data: dict) -> None:
        tmp = self.path.with_suffix(".tmp")
        tmp.write_text(json.dumps(data, indent=2), encoding="utf-8")
        os.replace(tmp, self.path)

    # ---- users ----
    def get(self, uid: str) -> Optional[dict]:
        return self._load()["users"].get(uid)

    def upsert(self, uid: str, **fields) -> dict:
        d = self._load()
        rec = d["users"].setdefault(uid, {"user_id": uid,
                                          "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                                          "status": "PENDING"})
        rec.update(fields)
        self._save(d)
        return rec

    def active_users(self) -> List[str]:
        return sorted(u for u, r in self._load()["users"].items() if r.get("status") == "ACTIVE")

    def fp_dir(self, uid: str) -> Path:
        return self.root / "fingerprints" / uid

    # ---- lockout ----
    def locked_for(self, key: str) -> float:
        rec = self._load()["lockouts"].get(key)
        return max(0.0, rec["locked_until"] - time.time()) if rec else 0.0

    def register_failure(self, key: str) -> None:
        d = self._load()
        rec = d["lockouts"].setdefault(key, {"fails": 0, "locked_until": 0.0})
        rec["fails"] += 1
        if rec["fails"] >= SYS.MAX_FAILS:
            rec["locked_until"] = time.time() + SYS.LOCKOUT_SEC
            rec["fails"] = 0
        self._save(d)

    def clear_failures(self, key: str) -> None:
        d = self._load()
        if d["lockouts"].pop(key, None) is not None:
            self._save(d)

    # ---- audit ----
    def log(self, event: str, user: Optional[str], **detail) -> None:
        new = not self.audit_path.is_file()
        with self.audit_path.open("a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            if new:
                w.writerow(["time", "event", "user", "detail"])
            w.writerow([time.strftime("%Y-%m-%d %H:%M:%S"), event, user or "",
                        json.dumps(detail, default=float)])

REGISTRY = Registry(SYS.DATA_DIR)

_MATCHER: Optional["SignatureMatcher"] = None

def get_matcher() -> "SignatureMatcher":
    """Load the encoder only once (avoid reloading the TF weights repeatedly)."""
    global _MATCHER
    if _MATCHER is None:
        _MATCHER = SignatureMatcher(SIG_CFG, SiameseEncoder(SIG_CFG.siamese_weights, SIG_CFG.img_w, SIG_CFG.img_h))
    return _MATCHER

In [ ]:
# SECTION 8: Signup = enrollment (fingerprint + air signature -> bind -> ACTIVE)
def load_fp_templates(uid: str) -> List[dict]:
    d = REGISTRY.fp_dir(uid)
    return [json.loads(p.read_text(encoding="utf-8")) for p in sorted(d.glob("template_*.json"))] if d.exists() else []

def find_fingerprint_owner(tpl: dict, exclude: str = "") -> Optional[Tuple[str, float]]:
    best = None
    for other in REGISTRY.active_users():
        if other == exclude:
            continue
        for t in load_fp_templates(other):
            r = match_fp(tpl, t)
            if fp_accept(r) and (best is None or r["score"] > best[1]):
                best = (other, r["score"])
    return best

def enroll_fingerprint(user_id: str, image_paths: Sequence[str]) -> dict:
    uid = sanitize_user(user_id)
    paths = [str(p) for p in image_paths]
    tpls, problems = [], []
    for p in paths:
        try:
            t = build_fp_template(p)
        except Exception as exc:
            problems.append(f"{os.path.basename(p)}: {exc}")
            continue
        bad = fp_quality_problems(t)
        if bad:
            problems.append(f"{os.path.basename(p)}: " + "; ".join(bad))
        else:
            tpls.append(t)

    if len({t["sha256"] for t in tpls}) != len(tpls):
        return dict(ok=False, reason="the same image file was given twice - provide different captures")
    if len(tpls) < SYS.FP_MIN_ENROLL_SAMPLES:
        return dict(ok=False, reason=f"at least {SYS.FP_MIN_ENROLL_SAMPLES} good fingerprint sample(s) required "
                                     f"(got {len(tpls)})", problems=problems)

    pair_scores = [match_fp(a, b)["score"] for a, b in combinations(tpls, 2)]
    if pair_scores and min(pair_scores) < SYS.FP_ENROLL_CONSISTENCY * SYS.FP_THRESHOLD:
        return dict(ok=False, reason="enrollment samples do not match each other (different finger, or a very poor capture)",
                    pair_scores=[round(s, 3) for s in pair_scores], problems=problems)

    owner = find_fingerprint_owner(tpls[0], exclude=uid)
    if owner:
        print(f"[warn] this finger seems to be already enrolled under user '{owner[0]}' (score {owner[1]:.2f})")

    d = REGISTRY.fp_dir(uid)
    d.mkdir(parents=True, exist_ok=True)
    for old in d.glob("template_*.json"):
        old.unlink()
    for i, t in enumerate(tpls, 1):
        (d / f"template_{i:02d}.json").write_text(json.dumps(t), encoding="utf-8")
    REGISTRY.upsert(uid, fingerprint=dict(enrolled=True, samples=len(tpls),
                                          sources=[t["source"] for t in tpls],
                                          paths=[t.get("path", "") for t in tpls],
                                          enrolled_at=time.strftime("%Y-%m-%d %H:%M:%S")))
    return dict(ok=True, samples=len(tpls), problems=problems,
                min_pair_score=round(min(pair_scores), 3) if pair_scores else None)

def enroll_signature(user_id: str) -> dict:
    uid = sanitize_user(user_id)
    return AirSignatureApp(SIG_CFG, uid, "ENROLL", matcher=get_matcher()).run()

def signup(user_id: str, fingerprint_images: Sequence[str], overwrite: bool = False) -> dict:
    uid = sanitize_user(user_id)
    rec = REGISTRY.get(uid)
    if rec and rec.get("status") == "ACTIVE" and not overwrite:
        print(f"[signup] user '{uid}' is already ACTIVE. Pass overwrite=True to enroll again.")
        return dict(ok=False, reason="user already exists")

    fp = enroll_fingerprint(uid, fingerprint_images)
    if not fp["ok"]:
        REGISTRY.log("signup_failed", uid, stage="fingerprint", reason=fp.get("reason"))
        print(f"[signup] fingerprint enrollment failed: {fp['reason']}")
        for p in fp.get("problems", []):
            print("   -", p)
        return dict(ok=False, stage="fingerprint", **{k: v for k, v in fp.items() if k != "ok"})
    REGISTRY.upsert(uid, status="PENDING")
    print(f"[signup] fingerprint enrolled ({fp['samples']} sample(s)). Now enroll the air signature...")

    sig = enroll_signature(uid)
    if not sig.get("ok"):
        REGISTRY.log("signup_failed", uid, stage="signature")
        print("[signup] air signature enrollment incomplete -> user stays PENDING (login not allowed)")
        return dict(ok=False, stage="signature")

    REGISTRY.upsert(uid, status="ACTIVE",
                    signature=dict(enrolled=True, samples=sig["samples"], threshold=sig["threshold"],
                                   enrolled_at=time.strftime("%Y-%m-%d %H:%M:%S")))
    REGISTRY.clear_failures(uid)
    REGISTRY.log("signup_ok", uid, fp_samples=fp["samples"], sig_samples=sig["samples"])
    print(f"[signup] '{uid}' is ACTIVE - fingerprint and air signature are now bound 1:1.")
    return dict(ok=True, user_id=uid)

In [ ]:
# SECTION 8b: Auto-enroll from a person_id/finger_id-structured database (hold back one sample per finger for later login testing)
SYS.FP_DATABASE_DIR = SYS.DATA_DIR.parent / "2D Database"

def find_person_finger_files(person_id, finger_id, image_dir=None):
    image_dir = Path(image_dir or SYS.FP_DATABASE_DIR)
    pat = re.compile(rf"^{int(person_id)}_{int(finger_id)}_(\d+)\.(bmp|png|jpg|jpeg|tif|tiff)$", re.IGNORECASE)
    if not image_dir.exists():
        raise FileNotFoundError(f"Dataset folder not found: {image_dir}")
    files = [f.name for f in image_dir.iterdir() if pat.match(f.name)]
    files.sort(key=lambda n: int(pat.match(n).group(1)))
    return image_dir, files

def signup_from_database(user_id, person_id, finger_id, image_dir=None,
                         n_enroll=None, overwrite=False):
    """Same as signup(), but the fingerprint images are picked from the database automatically.
    The air signature is still enrolled through the webcam (there is no dataset for it)."""
    uid = sanitize_user(user_id)
    image_dir, files = find_person_finger_files(person_id, finger_id, image_dir)
    if len(files) < SYS.FP_MIN_ENROLL_SAMPLES:
        print(f"[signup] person {person_id} finger {finger_id}: only {len(files)} samples found "
              f"({image_dir}), at least {SYS.FP_MIN_ENROLL_SAMPLES} required")
        return dict(ok=False, reason="not enough samples in database", found=files)

    n_enroll = n_enroll or max(SYS.FP_MIN_ENROLL_SAMPLES, len(files) - 1)
    n_enroll = min(n_enroll, len(files) - 1) if len(files) > SYS.FP_MIN_ENROLL_SAMPLES else len(files)
    enroll_files, held_out = files[:n_enroll], files[n_enroll:]

    print(f"[signup] person {person_id} finger {finger_id}: {len(files)} samples found -> "
          f"{len(enroll_files)} will be enrolled, held out (for login/testing): {held_out}")

    result = signup(uid, [str(image_dir / f) for f in enroll_files], overwrite=overwrite)
    result["held_out_files"] = [str(image_dir / f) for f in held_out]
    result["source"] = dict(person_id=int(person_id), finger_id=int(finger_id))
    return result

In [ ]:
# SECTION 8c: Auto-link from the fingerprint folder (fingerprint1.png ... fingerprintN.png, one sample per identity)
SYS.FP_MIN_ENROLL_SAMPLES = 1      # this dataset has only one image per identity

# Folder that contains fingerprint1.png, fingerprint2.png, ...
# Set the FP_POOL_DIR environment variable to point somewhere else without editing the code.
SYS.FP_POOL_DIR = Path(os.environ.get("FP_POOL_DIR", r"C:\Users\harma\Downloads\Fingerprint"))

# How login obtains the fingerprint when no image path is passed (automatic mode):
#   "same"     -> use the image linked at signup as the login capture (works with one sample per identity)
#   "simulate" -> use a slightly rotated/noisy copy of the linked image as a "fresh capture"
SYS.FP_AUTO_CAPTURE = "same"

def _pool_dir(image_dir=None) -> Path:
    image_dir = Path(image_dir or SYS.FP_POOL_DIR)
    if not image_dir.exists() and (BASE_DIR / "Fingerprint").exists():
        image_dir = BASE_DIR / "Fingerprint"          # fallback: a "Fingerprint" folder next to the notebook
    if not image_dir.exists():
        raise FileNotFoundError(f"Fingerprint folder not found: {image_dir}")
    return image_dir

def list_fingerprint_pool(image_dir=None):
    image_dir = _pool_dir(image_dir)
    pat = re.compile(r"^fingerprint(\d+)\.(png|bmp|jpg|jpeg|tif|tiff)$", re.IGNORECASE)
    files = [f.name for f in image_dir.iterdir() if pat.match(f.name)]
    files.sort(key=lambda n: int(pat.match(n).group(1)))
    return image_dir, files

def _linked_pool_hashes(exclude_uid=None):
    """Hashes of the images that are already linked to a user (so they are never handed to anyone else).
    exclude_uid: ignore this user's own earlier link, so a retry does not use up another image."""
    hashes = set()
    for uid in REGISTRY._load()["users"]:
        if uid == exclude_uid:
            continue
        for t in load_fp_templates(uid):
            hashes.add(t["sha256"])
    return hashes

def signup_from_pool(user_id, image_path=None, image_dir=None, overwrite=False):
    """Link ONE fingerprint image from the folder to a user, permanently.
    If image_path is not given, the next image that is not linked to anyone is picked automatically.
    The air signature is still enrolled through the webcam (there is no image dataset for it)."""
    uid = sanitize_user(user_id)
    rec = REGISTRY.get(uid)
    if rec and rec.get("status") == "ACTIVE" and not overwrite:
        print(f"[signup] user '{uid}' is already ACTIVE. Pass overwrite=True to enroll again.")
        return dict(ok=False, reason="user already exists")

    image_dir, files = list_fingerprint_pool(image_dir)
    if image_path is None:
        used = _linked_pool_hashes(exclude_uid=uid)
        chosen = next((image_dir / f for f in files if _sha256_file(image_dir / f) not in used), None)
        if chosen is None:
            return dict(ok=False, reason="no unused image left in the fingerprint folder")
        image_path = str(chosen)

    print(f"[signup] fingerprint image linked to '{uid}': {Path(image_path).name}")
    result = signup(uid, [str(image_path)], overwrite=overwrite)
    result["linked_file"] = str(image_path)
    return result

def linked_fingerprint_image(uid):
    """Return the path of the fingerprint image that was linked to this user at signup
    (None if the file cannot be found)."""
    fp = (REGISTRY.get(uid) or {}).get("fingerprint", {})
    candidates = [Path(p) for p in fp.get("paths", []) if p]
    try:
        candidates += [_pool_dir() / s for s in fp.get("sources", [])]
    except FileNotFoundError:
        pass
    return next((str(p) for p in candidates if p.is_file()), None)

def make_test_capture(image_path, out_path=None, seed=0):
    """DEMO/TESTING ONLY: builds a "fresh capture"-like variant (light noise + small rotation)
    of an enrolled image, so the login pipeline can be tested - this dataset has only ONE sample
    per identity, so no second genuine capture exists.
    This is NOT a real biometric verification test, it only checks that the pipeline works."""
    img = load_gray(str(image_path))
    rng = np.random.default_rng(seed)
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-4, 4), 1.0)
    out = cv2.warpAffine(img, M, (w, h), borderValue=255)
    out = np.clip(out.astype(np.float32) + rng.normal(0, 5, out.shape), 0, 255).astype(np.uint8)
    out_path = out_path or (str(Path(image_path).with_name(Path(image_path).stem + "_testcapture.png")))
    cv2.imwrite(out_path, out)
    return out_path

In [ ]:
# SECTION 9: Login = 2FA verification
def load_signature_profiles(uids: Sequence[str]) -> Dict[str, "Profile"]:
    matcher, profiles = get_matcher(), {}
    for uid in uids:
        samples = EnrollmentStore(SIG_CFG.enroll_dir, uid).load_samples()
        if samples:
            profiles[uid] = matcher.build_profile([t for t, _ in samples],
                                                  [c for _, c in samples if c is not None])
    return profiles

def signature_stage(claimed: Optional[str]) -> dict:
    """Verify the air signature via webcam (claimed user) or identify the user (unknown user)."""
    if claimed:
        app = AirSignatureApp(SIG_CFG, claimed, "VERIFY", matcher=get_matcher())
    else:
        profiles = load_signature_profiles(REGISTRY.active_users())
        app = AirSignatureApp(SIG_CFG, None, "IDENTIFY", profiles=profiles, matcher=get_matcher())
    return app.run()

def verify_fingerprint(uid: str, image_path: str, allow_identical: bool = False) -> dict:
    """1:1 fingerprint check against the templates of this user.
    allow_identical: accept a query image that is byte-identical to an enrolment image
    (only used by automatic login, where the linked image itself is the capture)."""
    templates = load_fp_templates(uid)
    if not templates:
        return dict(ok=False, reason="no fingerprint template", score=0.0, pairs=0)
    try:
        q = build_fp_template(image_path)
    except Exception as exc:
        return dict(ok=False, reason=f"image read/process error: {exc}", score=0.0, pairs=0)

    if not allow_identical and q["sha256"] in {t["sha256"] for t in templates}:
        return dict(ok=False, reason="identical to an enrolment image (a fresh capture is required)", score=0.0, pairs=0)
    bad = fp_quality_problems(q)
    if bad:
        return dict(ok=False, reason="poor quality: " + "; ".join(bad), score=0.0, pairs=0)

    best = max((match_fp(q, t) for t in templates), key=lambda r: r["score"])
    return dict(ok=fp_accept(best), reason="ok" if fp_accept(best) else "below threshold",
                score=best["score"], pairs=best["pairs"], rotation_deg=best["rotation_deg"])

def get_auto_capture(uid: str) -> Tuple[Optional[str], bool]:
    """Automatic fingerprint 'capture' for login: fetch the image that was linked to this user at signup.
    Returns (image_path, allow_identical); image_path is None if the linked image is missing."""
    linked = linked_fingerprint_image(uid)
    if linked is None:
        return None, False
    if SYS.FP_AUTO_CAPTURE == "simulate":
        out_dir = SYS.DATA_DIR / "tmp_captures"
        out_dir.mkdir(parents=True, exist_ok=True)
        capture = make_test_capture(linked, str(out_dir / f"{uid}_capture.png"), seed=int(time.time()) % 100000)
        return capture, False
    return linked, True

def on_access_granted(uid: str) -> None:
    """Hook: connect the locker unlock here (Arduino/ESP32 relay), e.g. serial.write(b'UNLOCK\\n')."""
    print(f"[access] UNLOCK signal -> user '{uid}'")

def _deny(reason_internal: str, uid: Optional[str], **info) -> dict:
    REGISTRY.log("login_denied", uid, reason=reason_internal, **info)
    print("ACCESS DENIED" + (f"  ({reason_internal})" if SYS.DEBUG else ""))
    return dict(granted=False, user_id=None, reason=reason_internal, **info)

def login(fingerprint_image: Optional[str] = None, user_id: Optional[str] = None) -> dict:
    """2FA login.
    fingerprint_image=None (default) -> the fingerprint linked at signup is loaded automatically.
    Pass a path only if you want to test a specific capture manually."""
    claimed = sanitize_user(user_id) if user_id else None
    key = claimed or "__anonymous__"

    wait = REGISTRY.locked_for(key)
    if wait > 0:
        print(f"Temporarily locked - try again in {wait:.0f}s")
        return dict(granted=False, user_id=None, reason="locked")

    if claimed:
        rec = REGISTRY.get(claimed)
        if not rec or rec.get("status") != "ACTIVE":
            REGISTRY.register_failure(key)
            return _deny("unknown or inactive user", claimed)
    elif not REGISTRY.active_users():
        return _deny("no active users enrolled", None)

    # ---------------- Stage 1: air signature ----------------
    sig = signature_stage(claimed)
    if not sig.get("ok"):
        REGISTRY.register_failure(key)
        return _deny("stage1 air signature failed", claimed, sig_score=sig.get("score"))
    uid = sig["user_id"]
    if REGISTRY.locked_for(uid) > 0:
        return _deny("account locked", uid)

    # ---------------- Stage 2: fingerprint (same user, 1:1) ----------------
    if fingerprint_image is None:
        fingerprint_image, allow_identical = get_auto_capture(uid)
        if fingerprint_image is None:
            return _deny("stage2 fingerprint failed: linked fingerprint image not found", uid,
                         sig_score=sig.get("score"))
        if SYS.DEBUG:
            print(f"[login] fingerprint loaded automatically: {Path(fingerprint_image).name}")
    else:
        fingerprint_image = str(fingerprint_image).strip().strip("\"'")   # tolerate quotes around a pasted path
        allow_identical = False

    fp = verify_fingerprint(uid, fingerprint_image, allow_identical=allow_identical)
    if not fp["ok"]:
        REGISTRY.register_failure(uid)
        if not claimed:
            REGISTRY.register_failure(key)
        return _deny(f"stage2 fingerprint failed: {fp['reason']}", uid,
                     sig_score=sig.get("score"), fp_score=fp["score"], fp_pairs=fp["pairs"])

    # ---------------- both passed ----------------
    REGISTRY.clear_failures(uid)
    REGISTRY.clear_failures(key)
    REGISTRY.log("login_granted", uid, sig_score=sig.get("score"), fp_score=fp["score"], fp_pairs=fp["pairs"])
    print(f"ACCESS GRANTED - welcome {uid}" + (f"   [sig {sig['score']:.2f} | fp {fp['score']:.2f} ({fp['pairs']} pairs)]" if SYS.DEBUG else ""))
    try:
        on_access_granted(uid)
    except Exception as exc:
        print(f"[warn] unlock hook failed: {exc}")
    return dict(granted=True, user_id=uid, sig_score=sig.get("score"), fp_score=fp["score"], fp_pairs=fp["pairs"])

In [ ]:
# SECTION 10: Registry viewer
def show_registry():
    users = REGISTRY._load()["users"]
    print(pd.DataFrame(users).T[["status"]] if users else "no users enrolled yet")
    if REGISTRY.audit_path.is_file():
        print(pd.read_csv(REGISTRY.audit_path).tail(10).to_string(index=False))

In [ ]:
# SECTION 11 (optional): Fingerprint threshold calibration (leakage-free, person-disjoint split)
import random

def parse_ids(fname):
    m = re.match(r"^(\d+)_(\d+)_(\d+)$", os.path.splitext(fname)[0])
    return tuple(int(v) for v in m.groups()) if m else None

def calibrate_fp_threshold(image_dir, target_far=0.01, fingers=(1,), max_persons=20, seed=42):
    files = [f for f in os.listdir(image_dir) if f.lower().endswith((".bmp", ".png", ".jpg", ".tif"))]
    groups = {}
    for f in files:
        ids = parse_ids(f)
        if ids and ids[1] in fingers:
            groups.setdefault((ids[0], ids[1]), []).append(f)
    persons = sorted({p for p, _ in groups})[:max_persons]
    calib_p = [p for i, p in enumerate(persons) if i % 2 == 0]
    test_p  = [p for i, p in enumerate(persons) if i % 2 == 1]

    cache = {}
    def tpl(f):
        if f not in cache:
            cache[f] = build_fp_template(os.path.join(image_dir, f))
        return cache[f]

    def scores(plist):
        rng = random.Random(seed)
        gen, imp = [], []
        keys = [k for k in groups if k[0] in plist]
        for k in keys:
            for a, b in combinations(sorted(groups[k]), 2):
                gen.append(match_fp(tpl(a), tpl(b))["score"])
        for _ in range(max(len(gen) * 10, 200)):
            k1, k2 = rng.sample(keys, 2)
            if k1[0] == k2[0]:
                continue
            imp.append(match_fp(tpl(rng.choice(groups[k1])), tpl(rng.choice(groups[k2])))["score"])
        return np.array(gen), np.array(imp)

    g_c, i_c = scores(calib_p)
    if len(i_c) < 1 / target_far:
        print(f"[warn] only {len(i_c)} impostor pairs -> the FAR={target_far} estimate is coarse (increase max_persons)")
    thr = float(np.quantile(i_c, 1 - target_far)) + 1e-6
    g_t, i_t = scores(test_p)
    out = dict(threshold=round(thr, 4),
               calib_FAR=float((i_c >= thr).mean()), calib_FRR=float((g_c < thr).mean()),
               test_FAR=float((i_t >= thr).mean()), test_FRR=float((g_t < thr).mean()),
               n_calib_persons=len(calib_p), n_test_persons=len(test_p),
               n_test_genuine=len(g_t), n_test_impostor=len(i_t))
    for k, v in out.items():
        print(f"{k:18s} {v}")
    print(f"\n=> SYS.FP_THRESHOLD = {out['threshold']}   (report only test_FAR / test_FRR)")
    return out

In [ ]:
# SECTION 12: Menu (run this cell to signup or login)
def run_menu():
    print("1. Signup")
    print("2. Login")
    print("3. View registry")
    choice = input("Choose an option: ").strip()

    if choice == "1":
        # The fingerprint is picked and linked automatically from the Fingerprint folder.
        uid = input("User ID: ").strip()
        result = signup_from_pool(uid)
        print(result)

    elif choice == "2":
        # No fingerprint path needed - the image linked at signup is loaded automatically.
        uid = input("User ID (leave blank to auto-identify via signature): ").strip() or None
        result = login(user_id=uid)
        print(result)

    elif choice == "3":
        show_registry()

    else:
        print("Invalid option")


run_menu()